# V15 Self-Play (Colab)

**Goal:** generate MORE self-play with the **new Q-recording recipe** — selfplay now records per-move `cand_prior` (clean pre-Dirichlet) + `cand_q` (root Q) + visits, the ingredients for the **Gumbel/advantage** target (plain visit-distillation prior-dominated and regressed). These fresh games **add to** the relabeled-V14 corpus (`relabel_v15`, clean-Q re-search); combined → the slim tensor → Gumbel trainer. NOTE: native self-play Q is from the Dirichlet-influenced search (slightly noisier than the clean relabel); fine as extra data, or relabel these too for uniform clean Q. Fresh seeds + dir.

**Player:** `pillar3f + value_head_pillar3f + q_weight=2.0` — current best (HISTORY 167-168). pillar3f = pillar3b + 0.5·(crisis task vector); 5k eval: **mean 31.6k, P50 22.2k, <1000 1.6%, max 301k**. value_head_pillar3f retrained on pillar3f's backbone per the HISTORY 158 rule (K-rollout r = 0.88/0.90/0.92/0.93 across H=25/50/100/200).

**Settings (V15, carried from the established plan):**
- **400 sims** (V13's budget; the prior is stronger, search refines + Dirichlet diversifies)
- q_weight=2.0 (the value-head calibration — HISTORY 133; the LINEAR evaluator would need 0.5)
- batch_size=8 (HISTORY 115)
- **max-turns=10000** (floor focus; NOTE: pillar3f's median game is ~10.5k turns, so roughly half the games will cap — raise to 15000 if late-game states look underrepresented in the first batch)
- **temperature-moves=10** (V15 plan: RNG provides spawn diversity; V13's 15 looked over-noisy early)
- Dirichlet α=0.3, weight=0.25 (defaults)

**Canary protocol:** run a small first shard (~20 seeds), check per-game scores land in/above pillar3f's policy-only ballpark (~22k median uncapped; capped games print `[CAPPED]`). If games come in far below, the head/q is misleading the search — stop and q-sweep before the long campaign.

## Upload to Drive (`MyDrive/alphatrain/`)

| file | size | source |
|---|---|---|
| `colorlines_pillar3d_v2.tar.gz` | ~450 KB | current code tarball (already on Drive) |
| `pillar3f.pt` | ~45 MB | local: `alphatrain/data/pillar3f.pt` |
| `value_head_pillar3f.pt` | ~38 KB | local: `alphatrain/data/value_head_pillar3f.pt` |

## Seed ranges (disjoint from existing data)

- V13 selfplay used 1300000-1300999
- V13 crisis used 800000-808000 (M5)
- **V15 selfplay**: 1700000+ (this notebook; shard across sessions/machines by sub-range)
- V15 crisis (M5, separate): 1800000+

## After this notebook

1. M5 generates V15 crisis in parallel (see Notes cell)
2. Build the V15 tensor — SLIM (`--policy-only-data` now drops the ~4GB value/pairwise fields and keeps `cand_prior`/`cand_q`):
   `build_expert_v2_tensor --games-dir data/selfplay_v15 data/crisis_v15 --policy-only-data --output alphatrain/data/v15_pillar3f.pt`
3. Train pillar3g with the **Gumbel/advantage trunk recipe** — target = softmax(prior z + root Q), disagreement-weighting (γ≈10), KL-anchor to pillar3f, **NO sharpening**. (Trainer is the next build.) Do NOT use the old visit-distillation + T=0.5 + aux-retention recipe — that prior-dominated and regressed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_pillar3d_v2.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v2.tar.gz

# GUARD: the uploaded tarball must be the Q-recording build, else self-play
# silently produces no-Q games (the exact thing we're fixing) — wasting the run.
assert 'last_root_record' in open('/content/alphatrain/scripts/selfplay.py').read(), \
    'STALE TARBALL: re-upload the Q-recording colorlines_pillar3d_v2.tar.gz (must have last_root_record).'
print('Q-recording recipe confirmed (selfplay records cand_prior/cand_q).')

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar3f.pt /content/alphatrain/data/pillar3f.pt
!cp {DRIVE}/value_head_pillar3f.pt /content/alphatrain/data/value_head_pillar3f.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3f.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_pillar3f.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V15 Self-Play ===
# Shard across Colab instances / the M5 by editing SEED_START/SEED_END.
# First session: run a SMALL canary shard (e.g. 1700000-1700020), check scores,
# then widen. V15 crisis (M5 local) uses 1800000+ — ranges stay disjoint.
SEED_START = 1700000
SEED_END = 1700020       # canary first; widen to e.g. 1700200 after
# =======================

SIMS = 400               # same as V13
Q_WEIGHT = 2.0           # value-head calibration (HISTORY 133); linear would be 0.5
WORKERS = 24             # Colab CPU
BATCH_SIZE = 8           # HISTORY 115
MAX_TURNS = 10000        # floor focus; ~half of pillar3f games will cap — see header note
TEMP_MOVES = 10          # V15 plan: RNG provides spawn diversity
SAVE_DIR = f'{DRIVE}/selfplay_v15'
# If a prior (aborted) run wrote games here, CLEAR them first — resume skips
# existing files, so stale no-Q games would silently stay in the corpus.

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, q_weight: {Q_WEIGHT}, cap: {MAX_TURNS} turns, temp_moves: {TEMP_MOVES}')
print(f'Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/pillar3f.pt \
    --value-head-path /content/alphatrain/data/value_head_pillar3f.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} \
    --max-turns {MAX_TURNS} \
    --compile

## Notes

- **Resume support:** if Colab disconnects, just re-run the previous cell. It skips seeds whose game files already exist in `SAVE_DIR` (which lives on Drive, so progress survives the VM).
- **Wall time:** pillar3f at MCTS@400 with cap=10K — stronger prior means longer games than V13's; expect ~3-6 min/game in 24-worker parallel. If a shard risks the 24h kill, split the seed range across instances.
- **Watch:** GPU server prints `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` every 10K forwards. avg bs should be ≥50 and evals/s in the thousands. If avg bs drops below 30, workers are starving — usually fine but worth flagging.
- **Output:** each game saved as `game_seed{N}_score{S}.json` in the save dir.
- **Crisis mining (M5 in parallel, seeds 1800000+):**
  ```
  caffeinate -dim python -m alphatrain.scripts.crisis_mining \
      --model alphatrain/data/pillar3f.pt \
      --value-head-path alphatrain/data/value_head_pillar3f.pt \
      --q-weight 2.0 \
      --seed-start 1800000 --seed-end 1801000 \
      --recovery-turns 15 --recovery-sims 600 \
      --prevention-turns 30 --prevention-sims 400 \
      --continue-turns 500 \
      --policy-max-turns 12000 \
      --workers 16 --device mps --batch-size 8 \
      --max-turns 25000 \
      --save-dir data/crisis_v15
  ```

## After both run

Build the V15 tensor (M5, ~30 min) — SLIM (`--policy-only-data` drops the ~4GB
value/pairwise fields, keeps `cand_prior`/`cand_q`):
```
python -m alphatrain.scripts.build_expert_v2_tensor \
    --games-dir data/selfplay_v15 data/crisis_v15 \
    --policy-only-data \
    --output alphatrain/data/v15_pillar3f.pt
```

Then train pillar3g with the **Gumbel/advantage trunk recipe** (the trainer is the next
build — NOT train_path_b's visit-distillation, which prior-dominated and regressed):
- target = softmax(prior z + σ(root Q)) [Gumbel] or advantage-softmax((Q − Q@prior-argmax)/τ)
- disagreement-weight the decisive states (w = 1 + γ·1[argmax Q ≠ argmax z], γ≈10)
- KL-anchor to pillar3f on broad states; **no target sharpening**
- warm-start the trunk from **pillar3b** (the trained base), not the pillar3f merge

Decision criterion: pillar3g ≥ pillar3f on **median + floor (P10/P25)**, eval uncapped
(`eval_policy`, seeds 775000-779999) vs the pillar3f bar (median 22,181 / mean 31,617).